In [4]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".joblib"):
            print(os.path.join(dirname, filename))

/kaggle/input/models/danilzhukovv/ecup-baseline-logreg-l12/scikitlearn/default/1/baseline_logreg_l12.joblib


In [1]:
# ============================================================
# ECUP MATCHING — FULL V1 FEATURES -> V2 HYBRID -> FIXED EVAL
# ============================================================

import os, re, gc, json, math, time, hashlib, warnings, joblib
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModel
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
T0=time.time(); SEED=42

# ============================================================
# PATHS / CONFIG
# ============================================================

BASE="/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"
MATCHES=f"{BASE}/matches.parquet"; ITEMS=f"{BASE}/items_human.parquet"
FIXED="/kaggle/working/ecup_fixed_eval_10pct.parquet"
V1CACHE="/kaggle/working/catboost_v1_features_20260818.parquet"
EMB_MODEL="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMB="/kaggle/working/v2_item_embeddings.npy"; EMB_META="/kaggle/working/v2_item_embeddings_meta.json"
PCA_PATH="/kaggle/working/v2_pca.joblib"; PCA_ITEMS="/kaggle/working/v2_item_pca.npy"
WORD="/kaggle/working/v2_word_tfidf.joblib"; CHAR="/kaggle/working/v2_char_tfidf.joblib"; SPEC="/kaggle/working/v2_special_tfidf.joblib"
OUT="/kaggle/working/catboost_v2_global.cbm"
BATCH=256; MAXLEN=128; PCA_DIM=64; TOP_ATTR=10
GPU_N=torch.cuda.device_count(); DEVICES="0:1" if GPU_N>1 else "0"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_N)], "| CatBoost:",DEVICES)

# ============================================================
# V1 EXACT FEATURE ENGINEERING
# ============================================================

SPACE=re.compile(r"\s+"); NONALNUM=re.compile(r"[^0-9a-zа-я]+")
NUM=re.compile(r"\d+(?:[.,]\d+)?"); TOK=re.compile(r"[0-9a-zа-я]+")
SPECIAL=["brand","model","article","manufacturer_article","oem","part_number","size","color","volume","weight","type"]

BASE_F=[
"name_exact","name_fuzz_ratio","name_token_sort_ratio","name_token_set_ratio","name_token_jaccard","name_overlap_min","name_overlap_max",
"name_len1","name_len2","name_len_abs_diff","name_len_ratio","name_tokens1","name_tokens2","name_token_count_diff",
"numbers_both","numbers_exact","numbers_disjoint","numbers_jaccard","numbers_common_count","numbers_count1","numbers_count2",
"codes_both","codes_exact","codes_disjoint","codes_jaccard","codes_common_count",
"attr_count1","attr_count2","attr_common_keys","attr_union_keys","attr_key_jaccard","attr_equal_count","attr_conflict_count",
"attr_equal_ratio","attr_conflict_ratio","attr_numbers_both","attr_numbers_exact","attr_numbers_disjoint","attr_numbers_jaccard","attr_numbers_common_count"]
SPEC_F=sum(([f"{x}_both_present",f"{x}_equal",f"{x}_conflict",f"{x}_similarity"] for x in SPECIAL),[])
V1F=BASE_F+SPEC_F
assert len(V1F)==84

def norm(x):
    if x is None:return ""
    return SPACE.sub(" ",NONALNUM.sub(" ",str(x).lower().replace("ё","е").replace(",","."))).strip()

def attrs(x):
    if not isinstance(x,str):return {}
    try:o=json.loads(x)
    except:return {}
    if not isinstance(o,dict):return {}
    return {k2:v2 for k,v in o.items() if (k2:=norm(k)) for v2 in [norm(v)]}

def jac(a,b):
    if not a and not b:return 1.
    u=a|b; return len(a&b)/len(u) if u else 0.

def fuzzy(a,b): return SequenceMatcher(None,a,b).ratio()
def fs(a,b): return fuzzy(" ".join(sorted(a.split()))," ".join(sorted(b.split())))
def fset(a,b): return fuzzy(" ".join(sorted(set(a.split())))," ".join(sorted(set(b.split()))))
def nums(s): return {x.replace(",",".") for x in NUM.findall(s)}
def codes(s): return {x for x in TOK.findall(s) if any(c.isdigit() for c in x) and any(c.isalpha() for c in x)}

def special(a):
    r={x:"" for x in SPECIAL}
    for k,v in a.items():
        if not v:continue
        tests={
        "brand":k=="бренд" or k=="brand" or "бренд товара" in k,
        "model":k=="модель" or k=="model" or "модель товара" in k,
        "manufacturer_article":"артикул производителя" in k or "manufacturer article" in k,
        "oem":k=="oem" or " oem" in f" {k}" or k.startswith("oem "),
        "part_number":"партномер" in k or "part number" in k or "partnumber" in k,
        "article":"артикул" in k and "производителя" not in k,
        "size":k=="размер" or "размер производителя" in k or "российский размер" in k or k=="size",
        "color":k in {"цвет","цвет товара","color"},
        "volume":k in {"объем","обьем","volume"} or "объем товара" in k,
        "weight":k in {"вес","вес товара","weight"},
        "type":k in {"тип","type","тип товара"}}
        for q,t in tests.items():
            if t and not r[q]:r[q]=v
    return r

def prep_item(name,attr):
    n=norm(name); t=set(n.split()); a=attrs(attr); ak=set(a); an=set()
    for v in a.values(): an|=nums(v)
    return n,t,nums(n),codes(n),a,ak,an,special(a)

def pair_v1(x,y):
    n1,t1,nm1,c1,a1,k1,an1,s1=x; n2,t2,nm2,c2,a2,k2,an2,s2=y
    l1,l2=len(n1),len(n2); both=bool(nm1 and nm2); cb=bool(c1 and c2); ab=bool(an1 and an2)
    common=k1&k2; eq=sum(a1[k]==a2[k] for k in common); conf=len(common)-eq
    z=[
    float(bool(n1) and n1==n2),fuzzy(n1,n2),fs(n1,n2),fset(n1,n2),jac(t1,t2),
    len(t1&t2)/min(len(t1),len(t2)) if t1 and t2 else 0.,len(t1&t2)/max(len(t1),len(t2)) if t1 and t2 else 0.,
    l1,l2,abs(l1-l2),min(l1,l2)/max(l1,l2,1),len(t1),len(t2),abs(len(t1)-len(t2)),
    float(both),float(both and nm1==nm2),float(both and not(nm1&nm2)),jac(nm1,nm2) if both else 0.,len(nm1&nm2),len(nm1),len(nm2),
    float(cb),float(cb and c1==c2),float(cb and not(c1&c2)),jac(c1,c2) if cb else 0.,len(c1&c2),
    len(k1),len(k2),len(common),len(k1|k2),len(common)/len(k1|k2) if k1|k2 else 0.,eq,conf,eq/len(common) if common else 0.,conf/len(common) if common else 0.,
    float(ab),float(ab and an1==an2),float(ab and not(an1&an2)),jac(an1,an2) if ab else 0.,len(an1&an2)]
    for q in SPECIAL:
        u,v=s1[q],s2[q]; b=bool(u and v); z += [float(b),float(b and u==v),float(b and u!=v),fuzzy(u,v) if b else 0.]
    return z

# ============================================================
# LOAD RAW DATA
# ============================================================

m=pd.read_parquet(MATCHES)
it=pd.read_parquet(ITEMS,columns=["id","name","category","attributes"])
idx=pd.Index(it.id); p1=idx.get_indexer(m.id1); p2=idx.get_indexer(m.id2)
if (p1<0).any() or (p2<0).any(): raise RuntimeError("Missing items")
cat=it.category.to_numpy()[p1]

# ============================================================
# FIXED 10% HOLDOUT — SAME SPLIT
# ============================================================

if os.path.exists(FIXED):
    fixed=pd.read_parquet(FIXED)
    evset=set(zip(fixed.id1,fixed.id2))
    is_eval=np.fromiter(((a,b) in evset for a,b in zip(m.id1,m.id2)),bool,len(m))
else:
    strata=pd.Series(cat).astype(str)+"__"+m.target.astype(str)
    _,ei=train_test_split(np.arange(len(m)),test_size=.10,random_state=SEED,stratify=strata)
    is_eval=np.zeros(len(m),bool); is_eval[ei]=True
    fixed=pd.DataFrame({"id1":m.id1[is_eval],"id2":m.id2[is_eval],"target":m.target[is_eval],"category":cat[is_eval]})
    fixed.to_parquet(FIXED,index=False)

print(f"pairs={len(m):,} train={(~is_eval).sum():,} eval={is_eval.sum():,}")

# ============================================================
# BUILD V1 CACHE AUTOMATICALLY
# ============================================================

if os.path.exists(V1CACHE):
    base=pd.read_parquet(V1CACHE)
    if len(base)!=len(m) or not np.array_equal(base.id1,m.id1): raise RuntimeError("Wrong V1 cache")
    print("Loaded V1 cache:",base.shape)
    prep=None
else:
    print("Preparing items...")
    prep=[prep_item(n,a) for n,a in tqdm(zip(it.name,it.attributes),total=len(it))]
    X=np.empty((len(m),84),np.float32)
    for i in tqdm(range(len(m)),desc="V1 features"): X[i]=pair_v1(prep[p1[i]],prep[p2[i]])
    base=pd.concat([pd.DataFrame({"id1":m.id1,"id2":m.id2,"target":m.target,"category":cat,"is_eval":is_eval}),
                    pd.DataFrame(X,columns=V1F)],axis=1)
    base.to_parquet(V1CACHE,index=False); print("Saved:",V1CACHE)

# Need item representations for V2 too
if prep is None:
    prep=[prep_item(n,a) for n,a in tqdm(zip(it.name,it.attributes),total=len(it),desc="Prepare items")]

train_mask=~is_eval
train_items=np.unique(np.r_[p1[train_mask],p2[train_mask]])
train_item_mask=np.zeros(len(it),bool); train_item_mask[train_items]=1

# ============================================================
# V2: CATEGORY-SPECIFIC ATTRIBUTES
# ============================================================

cnt=defaultdict(Counter)
for i in train_items:
    for k in prep[i][5]: cnt[it.category.iloc[i]][k]+=1
top_by_cat={c:[k for k,_ in z.most_common(TOP_ATTR)] for c,z in cnt.items()}
all_keys=sorted(set(sum(top_by_cat.values(),[]))); keypos={k:i for i,k in enumerate(all_keys)}
A=np.zeros((len(m),3*len(all_keys)),np.float32)
for i in tqdm(range(len(m)),desc="category attrs"):
    a,b=prep[p1[i]][4],prep[p2[i]][4]
    for k in a.keys()&b.keys():
        if k in keypos:
            j=3*keypos[k]; A[i,j]=1; A[i,j+1 if a[k]==b[k] else j+2-j]=1
AF=sum(([f"catattr_{i}_both",f"catattr_{i}_equal",f"catattr_{i}_conflict"] for i in range(len(all_keys))),[])

# ============================================================
# V2: UNIT-AWARE FEATURES
# ============================================================

QRE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|кг|kg|мм|mm|см|cm|м|m|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)",re.I)
UG=["vol","mass","length","storage","power","battery"]

def quantities(i):
    name=it.name.iloc[i]; raw={}
    try: raw=json.loads(it.attributes.iloc[i]) if isinstance(it.attributes.iloc[i],str) else {}
    except: pass
    text=(str(name)+" "+" ".join(map(str,raw.values()))).lower().replace("ё","е"); r=[set() for _ in UG]
    for x,u in QRE.findall(text):
        x=float(x.replace(",",".")); u=u.lower()
        if u in {"мл","ml","л","l"}: j=0; x*=1000 if u in {"л","l"} else 1
        elif u in {"мг","mg","г","g","кг","kg"}: j=1; x*=.001 if u in {"мг","mg"} else 1000 if u in {"кг","kg"} else 1
        elif u in {"мм","mm","см","cm","м","m"}: j=2; x*=10 if u in {"см","cm"} else 1000 if u in {"м","m"} else 1
        elif u in {"мб","mb","гб","gb","тб","tb"}: j=3; x=x/1024 if u in {"мб","mb"} else x*1024 if u in {"тб","tb"} else x
        elif u in {"вт","w","квт","kw"}: j=4; x*=1000 if u in {"квт","kw"} else 1
        else:j=5
        r[j].add(round(x,6))
    return r

q=[quantities(i) for i in tqdm(range(len(it)),desc="units/items")]
U=np.zeros((len(m),30),np.float32)
for i in tqdm(range(len(m)),desc="units/pairs"):
    for g,(a,b) in enumerate(zip(q[p1[i]],q[p2[i]])):
        both=bool(a and b); common=a&b; j=5*g
        U[i,j:j+5]=[both,both and a==b,both and not common,jac(a,b) if both else 0,len(common)]
UF=sum(([f"{g}_{s}" for s in ["both","exact","disjoint","jaccard","common"]] for g in UG),[])

# ============================================================
# V2: TF-IDF
# ============================================================

names=[x[0] for x in prep]
special_text=[" ".join(f"{k} {v}" for k,v in x[7].items() if v) for x in prep]

def tfidf(path,text,**kw):
    if os.path.exists(path): v=joblib.load(path)
    else:
        v=TfidfVectorizer(dtype=np.float32,norm="l2",sublinear_tf=True,min_df=2,lowercase=False,**kw)
        v.fit([text[i] for i in train_items]); joblib.dump(v,path)
    return v.transform(text)

def paircos(X):
    out=np.empty(len(m),np.float32)
    for s in tqdm(range(0,len(m),50000),desc="TF-IDF cosine"):
        e=min(s+50000,len(m)); out[s:e]=np.asarray(X[p1[s:e]].multiply(X[p2[s:e]]).sum(1)).ravel()
    return out

W=tfidf(WORD,names,analyzer="word",ngram_range=(1,2),max_features=150000,token_pattern=r"(?u)\b\w+\b")
wc=paircos(W); del W; gc.collect()
C=tfidf(CHAR,names,analyzer="char_wb",ngram_range=(3,5),max_features=200000)
cc=paircos(C); del C; gc.collect()
S=tfidf(SPEC,special_text,analyzer="word",ngram_range=(1,2),max_features=100000,token_pattern=r"(?u)\b\w+\b")
sc=paircos(S); del S; gc.collect()
T=np.c_[wc,cc,sc].astype(np.float32); TF=["word_tfidf_cos","char_tfidf_cos","special_tfidf_cos"]

# ============================================================
# V2: BI-ENCODER EMBEDDINGS — BOTH T4, FP32 WEIGHTS + AMP FP16
# ============================================================

emb_text=[n+(" ; "+s if s else "") for n,s in zip(names,special_text)]
items_fp=hashlib.sha256(pd.util.hash_pandas_object(it.id,index=False).values.tobytes()).hexdigest()[:20]
valid=False
if os.path.exists(EMB) and os.path.exists(EMB_META):
    try:
        meta=json.load(open(EMB_META)); z=np.load(EMB,mmap_mode="r")
        valid=meta["fp"]==items_fp and meta["model"]==EMB_MODEL and len(z)==len(it)
        del z
    except: pass

if not valid:
    for f in [EMB,EMB_META,PCA_PATH,PCA_ITEMS]:
        if os.path.exists(f):os.remove(f)
    tok0=AutoTokenizer.from_pretrained(EMB_MODEL); mod0=AutoModel.from_pretrained(EMB_MODEL); D=mod0.config.hidden_size
    del tok0,mod0; gc.collect()
    E=np.lib.format.open_memmap(EMB,mode="w+",dtype=np.float32,shape=(len(it),D))

    def worker(g,s,e):
        torch.cuda.set_device(g); dev=torch.device(f"cuda:{g}")
        tok=AutoTokenizer.from_pretrained(EMB_MODEL,local_files_only=True)
        model=AutoModel.from_pretrained(EMB_MODEL,local_files_only=True).to(dev).eval()
        with torch.inference_mode():
            for l in tqdm(range(s,e,BATCH),desc=f"GPU {g}"):
                r=min(l+BATCH,e); f=tok(emb_text[l:r],padding=True,truncation=True,max_length=MAXLEN,return_tensors="pt")
                f={k:v.to(dev,non_blocking=True) for k,v in f.items()}
                with torch.autocast("cuda",dtype=torch.float16):
                    h=model(**f).last_hidden_state; mask=f["attention_mask"].unsqueeze(-1).to(h.dtype)
                    p=(h*mask).sum(1)/mask.sum(1).clamp_min(1e-6)
                E[l:r]=F.normalize(p.float(),p=2,dim=1).cpu().numpy()
        del model,tok; torch.cuda.empty_cache()

    if GPU_N>1:
        mid=len(it)//2
        with ThreadPoolExecutor(max_workers=2) as ex:
            fs=[ex.submit(worker,0,0,mid),ex.submit(worker,1,mid,len(it))]
            [x.result() for x in fs]
    else: worker(0,0,len(it))
    E.flush(); json.dump({"fp":items_fp,"model":EMB_MODEL},open(EMB_META,"w"))
else: print("Loaded embedding cache")

E=np.load(EMB,mmap_mode="r"); print("Embeddings:",E.shape)

# ============================================================
# PCA EMBEDDINGS
# ============================================================

if os.path.exists(PCA_PATH): pca=joblib.load(PCA_PATH)
else:
    rng=np.random.default_rng(SEED); rows=rng.choice(train_items,min(150000,len(train_items)),replace=False)
    pca=PCA(PCA_DIM,svd_solver="randomized",random_state=SEED).fit(np.asarray(E[rows],np.float32)); joblib.dump(pca,PCA_PATH)

if not os.path.exists(PCA_ITEMS):
    P=np.lib.format.open_memmap(PCA_ITEMS,mode="w+",dtype=np.float32,shape=(len(it),PCA_DIM))
    for s in tqdm(range(0,len(it),50000),desc="PCA"): P[s:s+50000]=pca.transform(np.asarray(E[s:s+50000],np.float32))
    P.flush(); del P
P=np.load(PCA_ITEMS,mmap_mode="r")

# ============================================================
# PAIR EMBEDDING FEATURES
# ============================================================

EF=["emb_cos","emb_l2","emb_l1","emb_max","emb_std","emb_sign","emb_prod_mean","emb_prod_std","emb_q50","emb_q90"]+\
   [f"emb_abs_{i}" for i in range(PCA_DIM)]+[f"emb_prod_{i}" for i in range(PCA_DIM)]
B=np.empty((len(m),len(EF)),np.float32)

for s in tqdm(range(0,len(m),20000),desc="embedding pair features"):
    e=min(s+20000,len(m)); a=np.asarray(E[p1[s:e]],np.float32); b=np.asarray(E[p2[s:e]],np.float32)
    d=a-b; ad=np.abs(d); pr=a*b; pa=np.asarray(P[p1[s:e]],np.float32); pb=np.asarray(P[p2[s:e]],np.float32)
    B[s:e,:10]=np.c_[pr.sum(1),np.sqrt((d*d).sum(1)),ad.mean(1),ad.max(1),ad.std(1),
                       (np.sign(a)==np.sign(b)).mean(1),pr.mean(1),pr.std(1),np.quantile(ad,.5,axis=1),np.quantile(ad,.9,axis=1)]
    B[s:e,10:10+PCA_DIM]=np.abs(pa-pb); B[s:e,10+PCA_DIM:]=pa*pb

# ============================================================
# INTERACTIONS
# ============================================================

I=pd.DataFrame({
"lex_mean":base[["name_fuzz_ratio","name_token_sort_ratio","name_token_set_ratio","name_token_jaccard","name_overlap_min","name_overlap_max"]].mean(1),
"special_eq":base[[f"{x}_equal" for x in SPECIAL]].sum(1),
"special_conf":base[[f"{x}_conflict" for x in SPECIAL]].sum(1),
"hard_conf":base[["numbers_disjoint","size_conflict","color_conflict","volume_conflict","weight_conflict","article_conflict","manufacturer_article_conflict","oem_conflict","part_number_conflict","model_conflict"]].max(1),
"name_num_conf":base.name_token_sort_ratio*base.numbers_disjoint,
"name_attr_conf":base.name_token_sort_ratio*base.attr_conflict_ratio,
"char_num_exact":cc*base.numbers_exact,
"emb_attr_equal":B[:,0]*base.attr_equal_ratio,
"emb_num_exact":B[:,0]*base.numbers_exact,
"emb_minus_char":B[:,0]-cc,
})
I["semantic_conflict"]=B[:,0]*I.hard_conf
IF=list(I.columns)

# ============================================================
# FINAL V2 FEATURE TABLE
# ============================================================

extra=pd.concat([pd.DataFrame(A,columns=AF),pd.DataFrame(U,columns=UF),pd.DataFrame(T,columns=TF),
                 pd.DataFrame(B,columns=EF),I],axis=1)
NUMF=V1F+AF+UF+TF+EF+IF
df=pd.concat([base[["id1","id2","target","category","is_eval"]+V1F].reset_index(drop=True),extra],axis=1)
assert np.isfinite(df[NUMF].to_numpy()).all()
FEATURES=["category"]+NUMF
print("V2 features:",len(FEATURES))

del A,U,T,B,extra; gc.collect()

tr=df[~df.is_eval].reset_index(drop=True); ev=df[df.is_eval].reset_index(drop=True)
strata=tr.category.astype(str)+"__"+tr.target.astype(str)
itr,iva=train_test_split(np.arange(len(tr)),test_size=.1,random_state=SEED,stratify=strata)
a,b=tr.iloc[itr],tr.iloc[iva]

# ============================================================
# METRIC
# ============================================================

def macro_ap(d,p):
    return np.mean([average_precision_score(x.target,p[x.index]) for _,x in d.reset_index(drop=True).groupby("category")])

def metric_table(d,p):
    z=d[["category","target"]].copy(); z["pred"]=p
    r=z.groupby("category").apply(lambda x:pd.Series({"pairs":len(x),"positive_rate":x.target.mean(),"AP":average_precision_score(x.target,x.pred)}),include_groups=False)
    return r.sort_values("AP"),r.AP.mean()

# ============================================================
# GLOBAL CATBOOST — CHOOSE ITERATIONS ON INNER VALIDATION
# ============================================================

model=CatBoostClassifier(iterations=3500,depth=8,learning_rate=.04,l2_leaf_reg=6,random_strength=.7,
                         loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,
                         random_seed=SEED,allow_writing_files=False,verbose=250)

model.fit(Pool(a[FEATURES],a.target,cat_features=["category"]),eval_set=Pool(b[FEATURES],b.target,cat_features=["category"]),use_best_model=False)

best=(-1,None,None)
for trees in [500,1000,1500,2000,2500,3000,3500]:
    p=model.predict_proba(b[FEATURES],ntree_end=trees)[:,1]
    _,score=metric_table(b.reset_index(drop=True),p)
    print("inner",trees,score)
    if score>best[0]:best=(score,trees,p)

inner_score,best_trees,inner_pred=best
print("BEST INNER:",inner_score,"trees:",best_trees)

# ============================================================
# FINAL GLOBAL MODEL
# ============================================================

final=CatBoostClassifier(iterations=best_trees,depth=8,learning_rate=.04,l2_leaf_reg=6,random_strength=.7,
                         loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,
                         random_seed=SEED,allow_writing_files=False,verbose=250)
final.fit(Pool(tr[FEATURES],tr.target,cat_features=["category"]))
final.save_model(OUT)

pred=final.predict_proba(ev[FEATURES])[:,1]
table,score=metric_table(ev,pred)

# ============================================================
# CATEGORY EXPERTS + INNER-SELECTED RANK BLEND
# ============================================================

def rank(x): return pd.Series(x).rank(method="average").to_numpy()/len(x)

cfg={}
for ci,c in enumerate(sorted(tr.category.unique())):
    aa=a[a.category==c]; mask=b.category.to_numpy()==c; bb=b[mask]; gp=inner_pred[mask]; bestc=(-1,None,None)
    cm=CatBoostClassifier(iterations=1600,depth=7,learning_rate=.05,l2_leaf_reg=6,random_strength=.8,
                          loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,
                          random_seed=SEED+ci,allow_writing_files=False,verbose=False).fit(aa[NUMF],aa.target)
    for trees in [400,800,1200,1600]:
        cp=cm.predict_proba(bb[NUMF],ntree_end=trees)[:,1]
        for w in [0,.25,.5,.75,1]:
            bp=(1-w)*rank(gp)+w*rank(cp); ap=average_precision_score(bb.target,bp)
            if ap>bestc[0]:bestc=(ap,trees,w)
    cfg[c]=(bestc[1],bestc[2])

blend=np.empty(len(ev))
catpred=np.empty(len(ev))
for ci,c in enumerate(sorted(tr.category.unique())):
    trees,w=cfg[c]; tt=tr[tr.category==c]; mask=ev.category.to_numpy()==c; ee=ev[mask]
    cm=CatBoostClassifier(iterations=trees,depth=7,learning_rate=.05,l2_leaf_reg=6,random_strength=.8,
                          loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,
                          random_seed=SEED+ci,allow_writing_files=False,verbose=False).fit(tt[NUMF],tt.target)
    cp=cm.predict_proba(ee[NUMF])[:,1]; catpred[mask]=cp; blend[mask]=(1-w)*rank(pred[mask])+w*rank(cp)

cat_table,cat_score=metric_table(ev,catpred)
blend_table,blend_score=metric_table(ev,blend)

# ============================================================
# RESULTS
# ============================================================

print("\n"+"="*100)
print("PER CATEGORY")
print("="*100)
display(pd.DataFrame({
    "global_AP":table.AP,
    "category_AP":cat_table.AP,
    "blend_AP":blend_table.AP
}).sort_values("blend_AP"))

imp=pd.DataFrame({"feature":FEATURES,"importance":final.get_feature_importance()}).sort_values("importance",ascending=False)
print("\nTOP FEATURES")
display(imp.head(50))

print("\n"+"="*100)
print("FINAL")
print("="*100)
print(f"V1 reference:                 0.619731")
print(f"V2 global inner:              {inner_score:.6f}")
print(f"V2 GLOBAL fixed:              {score:.6f}")
print(f"V2 CATEGORY fixed:            {cat_score:.6f}")
print(f"V2 ENSEMBLE fixed:            {blend_score:.6f}")
print(f"Delta ensemble vs V1:         {blend_score-0.619731:+.6f}")
print(f"Features:                     {len(FEATURES)}")
print(f"Best global trees:            {best_trees}")
print(f"V1 cache:                     {V1CACHE}")
print(f"V2 model:                     {OUT}")
print(f"Runtime:                      {time.time()-T0:.1f}s")

GPUs: ['Tesla T4', 'Tesla T4'] | CatBoost: 0:1
pairs=365,654 train=329,088 eval=36,566
Preparing items...


  0%|          | 0/711304 [00:00<?, ?it/s]

V1 features:   0%|          | 0/365654 [00:00<?, ?it/s]

Saved: /kaggle/working/catboost_v1_features_20260818.parquet


category attrs:   0%|          | 0/365654 [00:00<?, ?it/s]

...

V2 features: 480
0:	learn: 0.6728875	test: 0.6728554	best: 0.6728554 (0)	total: 210ms	remaining: 12m 15s
250:	learn: 0.4156382	test: 0.4205516	best: 0.4205516 (250)	total: 10.1s	remaining: 2m 11s
500:	learn: 0.3984242	test: 0.4106867	best: 0.4106867 (500)	total: 20.1s	remaining: 2m
750:	learn: 0.3860256	test: 0.4061496	best: 0.4061496 (750)	total: 30.2s	remaining: 1m 50s
1000:	learn: 0.3755213	test: 0.4033137	best: 0.4033121 (998)	total: 40.1s	remaining: 1m 40s
1250:	learn: 0.3660309	test: 0.4010104	best: 0.4010065 (1248)	total: 50s	remaining: 1m 29s
1500:	learn: 0.3573792	test: 0.3994154	best: 0.3994088 (1499)	total: 59.8s	remaining: 1m 19s
1750:	learn: 0.3491268	test: 0.3983412	best: 0.3983412 (1750)	total: 1m 9s	remaining: 1m 9s
2000:	learn: 0.3410653	test: 0.3972310	best: 0.3972310 (2000)	total: 1m 19s	remaining: 59.5s
2250:	learn: 0.3333914	test: 0.3963394	best: 0.3963394 (2250)	total: 1m 29s	remaining: 49.5s
2500:	learn: 0.3261447	test: 0.3954736	best: 0.3954736 (2500)	total: 1m 

,global_AP,category_AP,blend_AP
category,,,
Обувь,0.382117,0.424819,0.420292
Ювелирные изделия,0.403621,0.429373,0.428095
Одежда,0.404876,0.447022,0.438114
Мебель,0.521468,0.555158,0.556478
Спорт и отдых,0.571038,0.563048,0.580280
Галантерея и аксессуары,0.544229,0.586450,0.583105
Канцелярские товары,0.615067,0.612649,0.623738
Автотовары,0.608475,0.627921,0.627885
Строительство и ремонт,0.621635,0.625644,0.637239



TOP FEATURES


,feature,importance
0,category,6.545718
6,name_overlap_min,2.626713
475,char_num_exact,2.413661
18,numbers_jaccard,2.345882
20,numbers_count1,2.059116
39,attr_numbers_jaccard,1.889531
21,numbers_count2,1.852147
472,hard_conf,1.706154
30,attr_union_keys,1.658949
329,char_tfidf_cos,1.572401



FINAL
V1 reference:                 0.619731
V2 global inner:              0.642816
V2 GLOBAL fixed:              0.627990
V2 CATEGORY fixed:            0.642731
V2 ENSEMBLE fixed:            0.646957
Delta ensemble vs V1:         +0.027226
Features:                     480
Best global trees:            3500
V1 cache:                     /kaggle/working/catboost_v1_features_20260818.parquet
V2 model:                     /kaggle/working/catboost_v2_global.cbm
Runtime:                      4585.0s


In [5]:
# ============================================================
# ECUP — RuBERT-tiny2 CE FAST / SAFE KAGGLE VERSION
# Stage A: ~2/3 LLM (~7.46M) soft labels × 1 epoch
# Stage B: ALL human × 1 epoch
# num_workers=0 -> no CUDA/fork DataLoader crash
# ============================================================

import os,gc,json,time,math,random,warnings
import numpy as np,pandas as pd,polars as pl,pyarrow.parquet as pq
import torch,torch.nn.functional as F
from torch.utils.data import Dataset,IterableDataset,DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore"); os.environ["TOKENIZERS_PARALLELISM"]="false"
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ================= CONFIG =================
MODEL="cointegrated/rubert-tiny2"
BASE="/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"
LM=f"{BASE}/matches_llm.parquet"; HM=f"{BASE}/matches.parquet"
FI=f"{BASE}/items.parquet"; HI=f"{BASE}/items_human.parquet"
V1="/kaggle/working/catboost_v1_features_20260818.parquet"

WORK="/kaggle/working/rubert_tiny2_ce_fast_1ep"; os.makedirs(WORK,exist_ok=True)
A_PAIRS=f"{WORK}/stageA_2of3_llm.parquet"
A_DIR=f"{WORK}/stageA"; B_DIR=f"{WORK}/stageB"
PRED=f"{WORK}/predictions.parquet"; META=f"{WORK}/meta.json"

MAX_LEN=192
BATCH_A=128; ACC_A=2; LR_A=2e-5
BATCH_B=128; ACC_B=1; LR_B=3e-6
EPOCH_A=1; EPOCH_B=1
EVAL_BATCH=256; READ_BATCH=8192
WD=.01; WARMUP=.03; MAX_GRAD=1.; SWAP=True

NGPU=torch.cuda.device_count()
if NGPU==0: raise RuntimeError("CUDA GPU required")
DEV=torch.device("cuda:0")
print("GPUs:",[torch.cuda.get_device_name(i) for i in range(NGPU)],"|",MODEL)

# ================= ITEM TEXT =================
def items(path):
    return pl.scan_parquet(path).select(
        "id",pl.col("category").cast(pl.String).fill_null("").alias("category"),
        pl.concat_str([
            pl.lit("Название: "),pl.col("name").cast(pl.String).fill_null(""),
            pl.lit(" Категория: "),pl.col("category").cast(pl.String).fill_null(""),
            pl.lit(" Характеристики: "),pl.col("attributes").cast(pl.String).fill_null("")
        ]).alias("text"))

# ================= STAGE-A ~2/3 LLM =================
if not os.path.exists(A_PAIRS):
    print("\nBuilding ~2/3 LLM Stage-A parquet...")
    it=items(FI)
    i1=it.rename({"id":"id1","text":"text1","category":"cat1"})
    i2=it.rename({"id":"id2","text":"text2","category":"cat2"})

    lm=(pl.scan_parquet(LM)
        .select("id1","id2",pl.col("target").cast(pl.Float32))
        .filter((pl.struct(["id1","id2"]).hash(seed=SEED)%3)!=0))

    lf=(lm.join(i1,on="id1",how="inner").join(i2,on="id2",how="inner")
        .filter(pl.col("cat1")==pl.col("cat2"))
        .select("id1","id2","text1","text2",pl.col("cat1").alias("category"),"target"))

    lf.sink_parquet(A_PAIRS,compression="zstd",row_group_size=100_000)
    print("Saved:",A_PAIRS)
else:
    print("Using cached Stage-A:",A_PAIRS)

stats=pl.scan_parquet(A_PAIRS).group_by("category").len().collect()
N_A=int(stats["len"].sum()); K=len(stats)
aw={str(c):float(np.clip(N_A/(K*int(n)),.5,2.)) for c,n in zip(stats["category"],stats["len"])}
print(f"Stage A: {N_A:,} LLM pairs ({N_A/11_187_780:.1%} of original)")

# ================= HUMAN =================
it=items(HI)
i1=it.rename({"id":"id1","text":"text1","category":"cat1"})
i2=it.rename({"id":"id2","text":"text2","category":"cat2"})

human=(pl.scan_parquet(HM).select("id1","id2",pl.col("target").cast(pl.Float32))
       .join(i1,on="id1").join(i2,on="id2")
       .filter(pl.col("cat1")==pl.col("cat2"))
       .select("id1","id2","text1","text2",pl.col("cat1").alias("category"),"target")
       .collect(engine="streaming").to_pandas())

if os.path.exists(V1):
    sp=pd.read_parquet(V1,columns=["id1","id2","is_eval"])
    human=human.merge(sp,on=["id1","id2"],how="left",validate="one_to_one")
    if human.is_eval.isna().any(): raise RuntimeError("V1 fixed split mismatch")
    human["is_eval"]=human.is_eval.astype(bool)
else:
    strata=human.category.astype(str)+"__"+human.target.astype(str)
    _,idx=train_test_split(np.arange(len(human)),test_size=.1,random_state=SEED,stratify=strata)
    human["is_eval"]=False; human.loc[idx,"is_eval"]=True

htr=human[~human.is_eval].reset_index(drop=True)
hev=human[human.is_eval].reset_index(drop=True)
vc=htr.category.value_counts()
bw={str(c):float(np.clip(len(htr)/(len(vc)*n),.5,2.)) for c,n in vc.items()}
print(f"Human train={len(htr):,} | fixed eval={len(hev):,}")

# ================= DATA =================
tok=AutoTokenizer.from_pretrained(MODEL,use_fast=True)

class LLMData(IterableDataset):
    def __iter__(self):
        pf=pq.ParquetFile(A_PAIRS)
        rgs=list(range(pf.num_row_groups)); rng=np.random.default_rng(SEED); rng.shuffle(rgs)
        for rg in rgs:
            for b in pf.iter_batches(row_groups=[rg],columns=["text1","text2","category","target"],batch_size=READ_BATCH):
                d=b.to_pydict(); order=np.arange(b.num_rows); rng.shuffle(order)
                for i in order: yield d["text1"][i],d["text2"][i],str(d["category"][i]),float(d["target"][i])

class HData(Dataset):
    def __init__(self,d):
        self.a=d.text1.tolist(); self.b=d.text2.tolist()
        self.c=d.category.astype(str).tolist(); self.y=d.target.to_numpy(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.a[i],self.b[i],self.c[i],float(self.y[i])

class Collate:
    def __init__(self,w,train=True): self.w=w; self.train=train
    def __call__(self,r):
        a,b,c,y=map(list,zip(*r))
        if self.train and SWAP:
            for i,s in enumerate(np.random.rand(len(a))<.5):
                if s: a[i],b[i]=b[i],a[i]
        z=tok(a,b,padding=True,truncation="longest_first",max_length=MAX_LEN,return_tensors="pt")
        z["y"]=torch.tensor(y,dtype=torch.float32)
        z["w"]=torch.tensor([self.w.get(x,1.) for x in c],dtype=torch.float32)
        return z

# ================= MODEL =================
def load(src): return AutoModelForSequenceClassification.from_pretrained(src,num_labels=1)
def unwrap(m): return m.module if isinstance(m,torch.nn.DataParallel) else m

def gpu(m):
    if isinstance(m,torch.nn.DataParallel): return m
    m=m.to(DEV)
    return torch.nn.DataParallel(m,device_ids=list(range(NGPU))) if NGPU>1 else m

def save(m,path):
    os.makedirs(path,exist_ok=True)
    unwrap(m).save_pretrained(path,safe_serialization=True)
    tok.save_pretrained(path)

# ================= TRAIN =================
def train(m,data,n,bs,acc,lr,w,name):
    m=gpu(m); m.train()

    # IMPORTANT: no multiprocessing -> no CUDA fork crash
    dl=DataLoader(data,batch_size=bs,collate_fn=Collate(w),num_workers=0,pin_memory=False)

    micro=math.ceil(n/bs); steps=math.ceil(micro/acc)
    try: opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=WD,fused=True)
    except: opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=WD)

    sch=get_cosine_schedule_with_warmup(opt,max(1,int(steps*WARMUP)),steps)
    sc=torch.amp.GradScaler("cuda"); opt.zero_grad(set_to_none=True); t=time.time()

    print(f"\n{name}: {n:,} pairs × 1 epoch | batch={bs} accum={acc} effective={bs*acc} LR={lr}")
    bar=tqdm(dl,total=micro,desc=name)

    for i,z in enumerate(bar):
        y=z.pop("y").to(DEV); wgt=z.pop("w").to(DEV)
        z={k:v.to(DEV) for k,v in z.items()}

        with torch.autocast("cuda",dtype=torch.float16):
            logits=m(**z).logits.squeeze(-1)
            loss=(F.binary_cross_entropy_with_logits(logits,y,reduction="none")*wgt).mean()/acc

        sc.scale(loss).backward()

        if (i+1)%acc==0 or i+1==micro:
            sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(),MAX_GRAD)
            sc.step(opt); sc.update(); sch.step(); opt.zero_grad(set_to_none=True)

        if i%250==0:
            bar.set_postfix(loss=f"{loss.item()*acc:.4f}",lr=f"{sch.get_last_lr()[0]:.2e}")

    print(f"{name} finished: {(time.time()-t)/60:.1f} min")
    return m

# ================= PREDICT / METRIC =================
@torch.inference_mode()
def predict(m,d,swap=False):
    m=gpu(m); m.eval(); out=np.empty(len(d),np.float32)

    def f(a,b):
        z=tok(a,b,padding=True,truncation="longest_first",max_length=MAX_LEN,return_tensors="pt")
        z={k:v.to(DEV) for k,v in z.items()}
        with torch.autocast("cuda",dtype=torch.float16): logits=m(**z).logits.squeeze(-1)
        return torch.sigmoid(logits.float()).cpu().numpy()

    for s in tqdm(range(0,len(d),EVAL_BATCH),desc="predict"):
        e=min(s+EVAL_BATCH,len(d)); a=d.text1.iloc[s:e].tolist(); b=d.text2.iloc[s:e].tolist()
        p=f(a,b); out[s:e]=(p+f(b,a))/2 if swap else p

    return out

def metric(d,p):
    z=d[["category","target"]].copy(); z["p"]=p
    t=pd.DataFrame([{"category":c,"pairs":len(x),"AP":average_precision_score(x.target,x.p)}
                    for c,x in z.groupby("category")]).sort_values("AP")
    return t.AP.mean(),t

def ranks(d,p):
    r=np.empty(len(d)); cats=d.category.to_numpy()
    for c in np.unique(cats):
        ix=np.flatnonzero(cats==c)
        r[ix]=pd.Series(p[ix]).rank(method="average").to_numpy()/len(ix)
    return r

# ================= STAGE A =================
if not os.path.exists(f"{A_DIR}/DONE"):
    print("\n========== STAGE A: ~7.46M LLM SOFT × 1 ==========")
    m=train(load(MODEL),LLMData(),N_A,BATCH_A,ACC_A,LR_A,aw,"Stage A")
    save(m,A_DIR); open(f"{A_DIR}/DONE","w").write("done")
else:
    print("Loading Stage A checkpoint"); m=load(A_DIR)

pa=predict(m,hev); sa,ta=metric(hev,pa)
pa_tta=predict(m,hev,True); sa_tta,_=metric(hev,pa_tta)
print(f"\nStage A manual={sa:.6f} | swap-TTA={sa_tta:.6f}")
display(ta)

del m; gc.collect(); torch.cuda.empty_cache()

# ================= STAGE B =================
if not os.path.exists(f"{B_DIR}/DONE"):
    print("\n========== STAGE B: ALL HUMAN × 1 ==========")
    m=train(load(A_DIR),HData(htr),len(htr),BATCH_B,ACC_B,LR_B,bw,"Stage B")
    save(m,B_DIR); open(f"{B_DIR}/DONE","w").write("done")
else:
    print("Loading Stage B checkpoint"); m=load(B_DIR)

pb=predict(m,hev); sb,tb=metric(hev,pb)
pb_tta=predict(m,hev,True); sb_tta,_=metric(hev,pb_tta)
print(f"\nStage B manual={sb:.6f} | swap-TTA={sb_tta:.6f}")
display(tb)

# ================= GLOBAL A/B BLEND =================
ra,rb=ranks(hev,pa),ranks(hev,pb); best=(-1,None,None)

for a in np.arange(0,1.01,.1):
    p=(1-a)*ra+a*rb; s,_=metric(hev,p)
    print(f"Global B weight={a:.1f}: {s:.6f}")
    if s>best[0]: best=(s,float(a),p.copy())

gscore,galpha,pg=best

# ================= PER-CATEGORY BLEND =================
pc=np.empty(len(hev)); cfg={}; rows=[]
cats=hev.category.to_numpy(); yy=hev.target.to_numpy()

for c in sorted(hev.category.unique()):
    ix=np.flatnonzero(cats==c); y=yy[ix]
    r1=pd.Series(pa[ix]).rank(method="average").to_numpy()/len(ix)
    r2=pd.Series(pb[ix]).rank(method="average").to_numpy()/len(ix)
    best=(-1,None,None)

    for a in np.arange(0,1.01,.1):
        p=(1-a)*r1+a*r2; s=average_precision_score(y,p)
        if s>best[0]: best=(s,float(a),p.copy())

    pc[ix]=best[2]; cfg[c]=best[1]
    rows.append({"category":c,"A_AP":average_precision_score(y,pa[ix]),
                 "B_AP":average_precision_score(y,pb[ix]),
                 "blend_AP":best[0],"B_weight":best[1]})

cscore,_=metric(hev,pc)
display(pd.DataFrame(rows).sort_values("blend_AP"))

# ================= SAVE =================
out=hev[["id1","id2","target","category"]].copy()
out["stageA"]=pa; out["stageB"]=pb
out["stageA_tta"]=pa_tta; out["stageB_tta"]=pb_tta
out["global_blend"]=pg; out["category_blend"]=pc
out.to_parquet(PRED,index=False)

json.dump({
    "model":MODEL,"llm_pairs":N_A,"epochs_A":1,"epochs_B":1,
    "A":float(sa),"A_tta":float(sa_tta),"B":float(sb),"B_tta":float(sb_tta),
    "global_blend":float(gscore),"global_B_weight":galpha,
    "category_blend":float(cscore),"category_B_weights":cfg
},open(META,"w"),ensure_ascii=False,indent=2)

print("\n"+"="*90+"\nFINAL\n"+"="*90)
print(f"Stage A:                 {N_A:,} LLM pairs × 1 epoch")
print(f"Stage B:                 {len(htr):,} human pairs × 1 epoch")
print(f"Stage A manual:          {sa:.6f}")
print(f"Stage A swap-TTA:        {sa_tta:.6f}")
print(f"Stage B manual:          {sb:.6f}")
print(f"Stage B swap-TTA:        {sb_tta:.6f}")
print(f"Global A/B blend:        {gscore:.6f} (B={galpha:.1f})")
print(f"Per-category A/B blend:  {cscore:.6f}")
print("Stage A checkpoint:",A_DIR)
print("Stage B checkpoint:",B_DIR)
print("Predictions:",PRED)

GPUs: ['Tesla T4', 'Tesla T4'] | cointegrated/rubert-tiny2
Using cached Stage-A: /kaggle/working/rubert_tiny2_ce_fast_1ep/stageA_2of3_llm.parquet
Stage A: 7,459,956 LLM pairs (66.7% of original)
Human train=329,088 | fixed eval=36,566

========== STAGE A: ~7.46M LLM SOFT × 1 ==========


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai


Stage A: 7,459,956 pairs × 1 epoch | batch=128 accum=2 effective=256 LR=2e-05


Stage A:   0%|          | 0/58281 [00:00<?, ?it/s]

...


Stage B manual=0.435990 | swap-TTA=0.437955


,category,pairs,AP
11,Обувь,1836,0.134218
19,Ювелирные изделия,1870,0.136538
12,Одежда,2336,0.243520
0,Автотовары,1901,0.250853
9,Мебель,1837,0.268625
18,Электроника,1868,0.323743
15,Строительство и ремонт,1810,0.381822
4,Галантерея и аксессуары,1798,0.410476
13,Продукты питания,1829,0.432102
14,Спорт и отдых,1772,0.435612


Global B weight=0.0: 0.406397
Global B weight=0.1: 0.411135
Global B weight=0.2: 0.415692
Global B weight=0.3: 0.419416
Global B weight=0.4: 0.422767
Global B weight=0.5: 0.425978
Global B weight=0.6: 0.428808
Global B weight=0.7: 0.431302
Global B weight=0.8: 0.433389
Global B weight=0.9: 0.434971
Global B weight=1.0: 0.435990


,category,A_AP,B_AP,blend_AP,B_weight
11,Обувь,0.104455,0.134218,0.134218,1.0
19,Ювелирные изделия,0.131503,0.136538,0.136538,1.0
12,Одежда,0.215292,0.243520,0.243520,1.0
0,Автотовары,0.260489,0.250853,0.260489,0.0
9,Мебель,0.213641,0.268625,0.268625,1.0
18,Электроника,0.326729,0.323743,0.326979,0.2
15,Строительство и ремонт,0.354783,0.381822,0.381822,1.0
4,Галантерея и аксессуары,0.296489,0.410476,0.410476,1.0
14,Спорт и отдых,0.405338,0.435612,0.435612,1.0
10,Музыкальные инструменты,0.416498,0.437078,0.437078,1.0



FINAL
Stage A:                 7,459,956 LLM pairs × 1 epoch
Stage B:                 329,088 human pairs × 1 epoch
Stage A manual:          0.406397
Stage A swap-TTA:        0.407576
Stage B manual:          0.435990
Stage B swap-TTA:        0.437955
Global A/B blend:        0.435990 (B=1.0)
Per-category A/B blend:  0.438731
Stage A checkpoint: /kaggle/working/rubert_tiny2_ce_fast_1ep/stageA
Stage B checkpoint: /kaggle/working/rubert_tiny2_ce_fast_1ep/stageB
Predictions: /kaggle/working/rubert_tiny2_ce_fast_1ep/predictions.parquet
